In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


False

In [3]:
# colab-only
!pip install giskard-checks openai

Use `UserSimulator` to drive multi-turn tests with LLM-generated user inputs.

## 1. Configure a generator

To get started, provide the LLM that powers the simulator. `UserSimulator` uses
the default **generator** (`set_default_generator`) — see [Generators and
judges](/oss/checks/explanation/core-concepts#generators-and-judges). Set it
once here, or pass a generator inline.

In [3]:
def support_agent(message: str) -> str:
    """Stub support agent for demonstration."""
    return "I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?"

from giskard.checks import set_default_generator

set_default_generator("openai/gpt-5.4-nano")


Thank you for using Giskard open-source! 🐢 🙏
Giskard Enterprise adds deeper agent scans, audit reports
with remediation guidance, test review interfaces
for root-cause analysis & human feedback integration,
and team collaboration — with flexible pricing.
Learn more: https://giskard.ai


## 2. Create a `UserSimulator` with a persona

With the generator configured, we can now define who the simulated user is. The
`persona` field acts as a system prompt for the simulator — it describes the
user's role, goal, and stopping condition. The more specific you are, the more
deterministic and useful the generated conversation will be.

In [4]:
from giskard.checks.generators.user import UserSimulator

customer = UserSimulator(
    persona="""
    You are a customer trying to track a delayed order.
    - Start by asking about order #98765
    - Provide your name (Alex) when asked
    - Accept any resolution the support agent offers
    - Stop when the agent confirms a solution
    """,
    max_steps=8,
)

`max_steps` limits how many turns the simulator will generate before stopping.

## 3. Use the simulator as `inputs` in `.interact()`

Now we'll wire the simulator into the scenario. Passing the `UserSimulator` as
`inputs` tells the scenario to call it on each turn rather than using a fixed
string — the scenario handles the loop automatically up to `max_steps`.

Pass the `UserSimulator` instance as the `inputs` argument. The scenario will
call it repeatedly to generate each user turn.

In [5]:
from giskard.checks import Scenario, FnCheck

scenario = (
    Scenario("order_tracking")
    .interact(
        inputs=customer,
        outputs=lambda inputs: support_agent(inputs),
    )
    .check(
        FnCheck(fn=
            lambda trace: any(
                word in trace.last.outputs.lower()
                for word in ["resolved", "refund", "replacement", "shipped"]
            ),
            name="resolution_offered",
        )
    )
)

## 4. Run the scenario and inspect the trace

With the scenario built, run it and iterate over the trace to see the full
conversation the simulator generated. This is especially useful when debugging a
failing check — you can see exactly what the simulated user said at each step.

In [6]:
import asyncio
result = asyncio.run(scenario.run())

# Print every turn
for turn in result.final_trace.interactions:
    print(f"User:  {turn.inputs}")
    print(f"Agent: {turn.outputs}")
    print()

User:  Hi, I’m checking on a delayed order—could you look up order #98765 and tell me the current status and when it will ship?
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?

User:  Yes—my name is Alex. Can you confirm the updated tracking details for order #98765 and whether there’s any chance it could ship today instead of tomorrow?
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?

User:  Thanks—I'm Alex. Can you please confirm the latest tracking number/carrier details for order #98765 and what the updated estimated delivery date is? If there’s any option to expedite or a new ship date, please let me know.
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?

User:  I appreciate the update. Can you please provide the actu

## 5. Check `goal_reached` from simulator metadata

After the scenario finishes, the simulator writes a `LLMGeneratorOutput` into
the last interaction's metadata. This tells you whether the user's stated goal
was achieved, a stronger signal than just checking whether the scenario passed
its checks, because it reflects the simulator's own evaluation of the
conversation outcome.

In [7]:
from giskard.checks.generators.base import LLMGeneratorOutput

last = result.final_trace.last
simulator_output = last.metadata.get("simulator_output")

if isinstance(simulator_output, LLMGeneratorOutput):
    print(f"Goal reached: {simulator_output.goal_reached}")
    print(f"Message: {simulator_output.message}")

Use `goal_reached` as an additional assertion:

In [8]:
if simulator_output and not simulator_output.goal_reached:
    print(f"Goal not reached: {simulator_output.message}")
else:
    print("Goal reached or no simulator output")


Goal reached or no simulator output


## 6. Swap personas for A/B testing

With a single persona working, we can now run the same agent against multiple
user types simultaneously. Each persona exercises a different interaction style,
and running them concurrently with `asyncio.gather` means you get results for
all three in roughly the time it takes to complete one.

Run the same agent against multiple user types to surface persona-specific
failures.

In [9]:
import asyncio

personas = [
    (
        "impatient",
        "You are impatient. Keep messages short. Escalate quickly if not helped.",
    ),
    (
        "detailed",
        "You are thorough. Ask many follow-up questions before accepting any solution.",
    ),
    (
        "confused",
        "You are unsure what you need. Describe symptoms, not the actual problem.",
    ),
]


async def run_persona(name, instructions):
    sim = UserSimulator(persona=instructions, max_steps=6)
    scenario = Scenario(name).interact(
        inputs=sim,
        outputs=lambda inputs: support_agent(inputs),
    )
    return name, await scenario.run()


results = await asyncio.gather(*[run_persona(n, i) for n, i in personas])

for name, result in results:
    print(f"{name}: {'PASSED' if result.passed else 'FAILED'}")

impatient: PASSED
detailed: PASSED
confused: PASSED


## Custom trace formatting

By default the trace prints interactions as raw inputs and outputs. You can
write a simple formatting function to produce a human-readable transcript — for
example, to log a simulated conversation or include it in a test failure message.
For a subclass of `Trace`, Rich rendering, and how that interacts with
`print_report()`, see [Custom trace types](/oss/checks/how-to/custom-trace).

In [10]:
def format_transcript(trace) -> str:
    """Format a trace as a human-readable chat transcript."""
    lines = []
    for turn in trace.interactions:
        lines.append(f"User:  {turn.inputs}")
        lines.append(f"Agent: {turn.outputs}")
    return "\n".join(lines)


result = await (
    Scenario("chat_trace_demo")
    .interact(
        inputs=customer,
        outputs=lambda inputs: support_agent(inputs),
    )
    .run()
)

print(format_transcript(result.final_trace))

User:  Hi, I’m trying to track a delayed order. Can you look up order #98765 and tell me the current status and expected delivery date?
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?
User:  Thanks—what’s the latest tracking update for order #98765, and can you confirm the expected delivery date/time window for tomorrow?
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?
User:  Okay—can you please confirm the exact delivery time window for tomorrow, and whether there’s any updated tracking scan or exception? If it’s still delayed, what resolution can you offer (replacement/refund/shipping update)?
Agent: I have located your order #98765. It is currently in transit and will arrive tomorrow. Is there anything else I can help you with?
User:  Hi again—this is Alex. Can you please give me the confirmed delivery time

## Next steps

- [Generators reference](/oss/checks/reference/generators) — full
  `UserSimulator` parameter reference
- [Multi-turn testing tutorial](/oss/checks/tutorials/multi-turn) — multi-turn
  scenario basics
- [Debug with Spy](/oss/checks/how-to/spy-on-calls) — inspect what happens
  inside each interaction